In [27]:

import requests
import time
from pyspark.sql import SparkSession


# Đây chính là "Tờ hướng dẫn" hoàn hảo để giao cho Spark
def fetch_and_extract(url):
    print(f"Đang cào: {url}") # In ra để lúc test dễ nhìn
   
    try:
        response = requests.get(url, timeout=5)
       
        if response.status_code == 200:
            data = response.json()
            # Trả về từ điển chứa Data sạch
            return {
                "id": data.get("id"),
                "title": data.get("title", "Không có tên"),
                "price": data.get("price", 0.0),
                "brand": data.get("brand", "Unknown"),
                "rating": data.get("rating", 0.0)
            }
        else:
            # Trả về từ điển chứa Lỗi hệ thống
            return {"id": None, "error": f"Lỗi HTTP: {response.status_code}"}
           
    except requests.exceptions.Timeout:
        # Trả về từ điển chứa Lỗi chờ lâu
        return {"id": None, "error": "Lỗi: Quá thời gian chờ (Timeout)"}
       
    except requests.exceptions.RequestException as e:
        # Trả về từ điển chứa Lỗi sập mạng
        return {"id": None, "error": "Lỗi: Sập kết nối mạng"}


thoi_gian_bat_dau = time.time()
print("🚨 Quản đốc thổi còi: BẮT ĐẦU CHẠY!")
# SparkSession

# Import thư viện Spark (Đã được cài sẵn trong phòng thí nghiệm Docker của bạn)

print("Đang gọi Quản đốc Spark tới công trường...")

# 1. Khởi tạo Spark Session
spark = SparkSession.builder \
    .appName("Mini_DE_Crawler_Tiki") \
    .master("local[4]") \
    .getOrCreate()

# 2. Kiểm tra xem ông ấy đã có mặt chưa
print("✅ Quản đốc Spark đã sẵn sàng làm việc!")
print(f"🛠️ Phiên bản Spark đang dùng: {spark.version}")
####################################################################################################
# 1. Tạo ra một xấp 10 tờ giấy (10 URLs) bằng Python thuần
# Dùng vòng lặp for đơn giản để tạo link từ 1 đến 10
danh_sach_urls = [f"https://dummyjson.com/products/{i}" for i in range(1, 31)]

print(f"📦 Đã chuẩn bị xấp công việc: {len(danh_sach_urls)} links.")

# 2. Đưa cho quản đốc Spark để "xé nhỏ" (Chuyển List thành RDD)
# spark.sparkContext (hay sc) chính là cánh tay phải của ông quản đốc
urls_rdd = spark.sparkContext.parallelize(danh_sach_urls)

# 3. Kiểm tra xem quản đốc đã chia công việc ra làm mấy phần (partitions)?
so_phan_chia = urls_rdd.getNumPartitions()

print(f"✂️ Quản đốc Spark đã chia 10 links này thành {so_phan_chia} phần (partitions).")
#####################################################################################################
# Giao việc: Ép 4 công nhân dùng hàm fetch_and_extract cho từng đường link
ket_qua_rdd = urls_rdd.map(fetch_and_extract)


print("📝 Quản đốc: 'Đã phát xong tờ hướng dẫn cho 4 anh em!'")
print("Tình trạng của RDD hiện tại:")
print(ket_qua_rdd)


# 2. PHÉP THUẬT XẢY RA Ở ĐÂY: Dùng .collect() để kích hoạt chạy thật
# Lúc này 4 core CPU của máy bạn mới thực sự vắt chân lên cổ chạy tải data
danh_sach_ket_qua = ket_qua_rdd.collect()


# 3. Dừng đồng hồ
thoi_gian_ket_thuc = time.time()
tong_thoi_gian = thoi_gian_ket_thuc - thoi_gian_bat_dau


print(f"⏱️ THẦN TỐC: Đã cào xong 10 trang web chỉ trong {tong_thoi_gian:.2f} giây!")
print(f"📦 Tổng số món hàng thu được: {len(danh_sach_ket_qua)}")


# 4. Mở 2 thùng hàng đầu tiên ra kiểm tra xem data có đẹp không
print("\n--- HÀNG MẪU VỀ KHO ---")
for mon_hang in danh_sach_ket_qua:
    print(mon_hang)
print("Default Parallelism:", spark.sparkContext.defaultParallelism)

🚨 Quản đốc thổi còi: BẮT ĐẦU CHẠY!
Đang gọi Quản đốc Spark tới công trường...
✅ Quản đốc Spark đã sẵn sàng làm việc!
🛠️ Phiên bản Spark đang dùng: 3.5.0
📦 Đã chuẩn bị xấp công việc: 30 links.
✂️ Quản đốc Spark đã chia 10 links này thành 4 phần (partitions).
📝 Quản đốc: 'Đã phát xong tờ hướng dẫn cho 4 anh em!'
Tình trạng của RDD hiện tại:
PythonRDD[40] at RDD at PythonRDD.scala:53
⏱️ THẦN TỐC: Đã cào xong 10 trang web chỉ trong 4.75 giây!
📦 Tổng số món hàng thu được: 30

--- HÀNG MẪU VỀ KHO ---
{'id': 1, 'title': 'Essence Mascara Lash Princess', 'price': 9.99, 'brand': 'Essence', 'rating': 2.56}
{'id': 2, 'title': 'Eyeshadow Palette with Mirror', 'price': 19.99, 'brand': 'Glamour Beauty', 'rating': 2.86}
{'id': 3, 'title': 'Powder Canister', 'price': 14.99, 'brand': 'Velvet Touch', 'rating': 4.64}
{'id': 4, 'title': 'Red Lipstick', 'price': 12.99, 'brand': 'Chic Cosmetics', 'rating': 4.36}
{'id': 5, 'title': 'Red Nail Polish', 'price': 8.99, 'brand': 'Nail Couture', 'rating': 4.32}
{'i

In [24]:

import requests
import time

# Đây chính là "Tờ hướng dẫn" hoàn hảo để giao cho Spark
def fetch_and_extract(url):
    print(f"Đang cào: {url}", end = "...") # In ra để lúc test dễ nhìn
   
    try:
        response = requests.get(url, timeout=5)
       
        if response.status_code == 200:
            data = response.json()
            # Trả về từ điển chứa Data sạch
            return {
                "id": data.get("id"),
                "title": data.get("title", "Không có tên"),
                "price": data.get("price", 0.0),
                "brand": data.get("brand", "Unknown"),
                "rating": data.get("rating", 0.0)
            }
        else:
            # Trả về từ điển chứa Lỗi hệ thống
            return {"id": None, "error": f"Lỗi HTTP: {response.status_code}"}
           
    except requests.exceptions.Timeout:
        # Trả về từ điển chứa Lỗi chờ lâu
        return {"id": None, "error": "Lỗi: Quá thời gian chờ (Timeout)"}
       
    except requests.exceptions.RequestException as e:
        # Trả về từ điển chứa Lỗi sập mạng
        return {"id": None, "error": "Lỗi: Sập kết nối mạng"}


# 1. Bấm đồng hồ tính giờ
thoi_gian_bat_dau = time.time()
####################################################################################################
# 1. Tạo ra một xấp 10 tờ giấy (10 URLs) bằng Python thuần
# Dùng vòng lặp for đơn giản để tạo link từ 1 đến 10
danh_sach_urls = [f"https://dummyjson.com/products/{i}" for i in range(1, 31)]

print(f"📦 Đã chuẩn bị xấp công việc: {len(danh_sach_urls)} links.")

print("🚨 Quản đốc thổi còi: BẮT ĐẦU CHẠY!")

danh_sach_ket_qua = []
for url in danh_sach_urls:
    danh_sach_ket_qua.append(fetch_and_extract(url))

# 3. Dừng đồng hồ
thoi_gian_ket_thuc = time.time()
tong_thoi_gian = thoi_gian_ket_thuc - thoi_gian_bat_dau


print(f"⏱️ THẦN TỐC: Đã cào xong 10 trang web chỉ trong {tong_thoi_gian:.2f} giây!")
print(f"📦 Tổng số món hàng thu được: {len(danh_sach_ket_qua)}")


# 4. Mở 2 thùng hàng đầu tiên ra kiểm tra xem data có đẹp không
print("\n--- HÀNG MẪU VỀ KHO ---")
for mon_hang in danh_sach_ket_qua:
    print(mon_hang)


📦 Đã chuẩn bị xấp công việc: 30 links.
🚨 Quản đốc thổi còi: BẮT ĐẦU CHẠY!
Đang cào: https://dummyjson.com/products/1...Đang cào: https://dummyjson.com/products/2...Đang cào: https://dummyjson.com/products/3...Đang cào: https://dummyjson.com/products/4...Đang cào: https://dummyjson.com/products/5...Đang cào: https://dummyjson.com/products/6...Đang cào: https://dummyjson.com/products/7...Đang cào: https://dummyjson.com/products/8...Đang cào: https://dummyjson.com/products/9...Đang cào: https://dummyjson.com/products/10...Đang cào: https://dummyjson.com/products/11...Đang cào: https://dummyjson.com/products/12...Đang cào: https://dummyjson.com/products/13...Đang cào: https://dummyjson.com/products/14...Đang cào: https://dummyjson.com/products/15...Đang cào: https://dummyjson.com/products/16...Đang cào: https://dummyjson.com/products/17...Đang cào: https://dummyjson.com/products/18...Đang cào: https://dummyjson.com/products/19...Đang cào: https://dummyjson.com/products/20...Đang cào: https

In [26]:
import requests
import time
from pyspark.sql import SparkSession

# ==============================
# Hàm crawl
# ==============================
def fetch_and_extract(url):
    try:
        response = requests.get(url, timeout=5)

        if response.status_code == 200:
            data = response.json()
            return {
                "id": data.get("id"),
                "title": data.get("title"),
                "price": data.get("price"),
                "brand": data.get("brand"),
                "rating": data.get("rating")
            }
        else:
            return {"id": None, "error": f"HTTP {response.status_code}"}

    except Exception:
        return {"id": None, "error": "Request failed"}


# ==============================
# Hàm benchmark
# ==============================
def run_spark_job(num_partitions):
    print(f"\n🚀 Chạy với {num_partitions} partitions")

    start = time.time()

    urls = [f"https://dummyjson.com/products/{i}" for i in range(1, 31)]

    rdd = spark.sparkContext.parallelize(urls, num_partitions)

    print(f"👉 Partitions thực tế: {rdd.getNumPartitions()}")

    results = rdd.map(fetch_and_extract).collect()

    end = time.time()

    print(f"⏱️ Time: {end - start:.2f}s | Records: {len(results)}")

    return end - start


# ==============================
# Spark init (chỉ chạy 1 lần)
# ==============================
spark = SparkSession.builder \
    .appName("Compare_4_vs_8") \
    .master("local[8]") \
    .getOrCreate()

print("✅ Spark ready:", spark.version)


# ==============================
# So sánh
# ==============================
time_4 = run_spark_job(4)
time_8 = run_spark_job(8)

print("\n📊 KẾT QUẢ:")
print(f"4 partitions: {time_4:.2f}s")
print(f"8 partitions: {time_8:.2f}s")

✅ Spark ready: 3.5.0

🚀 Chạy với 4 partitions
👉 Partitions thực tế: 4
⏱️ Time: 5.70s | Records: 30

🚀 Chạy với 8 partitions
👉 Partitions thực tế: 8
⏱️ Time: 6.08s | Records: 30

📊 KẾT QUẢ:
4 partitions: 5.70s
8 partitions: 6.08s


NameError: name 'spark' is not defined